# L17 · 特征分解（eigendecomposition） A=PDP⁻¹ — 换坐标系让矩阵乘法变成标量乘法

**目标**：掌握 `Av = λv`、特征方程（characteristic equation） `det(A − λI) = 0`；对称矩阵（symmetric matrix）可对角化（diagonalization） `A = PΛP⁻¹`，矩阵幂次 `Aⁿ = PΛⁿP⁻¹` 退化为标量幂次。

**为什么对 Aurora 重要**：递归滤波器（recursive filter）和状态空间模型（state space model）的稳定性由系统矩阵的特征值（eigenvalue）决定——所有 `|λ| < 1` 则滤波器收敛；`aurora.dsp` 的极点分析依赖这套分解。

## 本课剧情：寻找不拐弯的方向

大多数向量经过矩阵会同时改变方向和长度；特征向量（eigenvector）只改变长度，方向不变，缩放比例就是特征值 `λ`。找到全部特征向量后，可以把矩阵写成 `A = PΛP⁻¹`，此时计算 `Aⁿ` 只需把对角线上各 `λ` 逐个乘方，矩阵运算降为标量运算。

## 1. 特征方程：`det(A − λI) = 0` 的根就是特征值（补充例题）

补充例题：$A=\begin{pmatrix}3&3&3\\3&-1&1\\3&1&-1\end{pmatrix}$，特征值 $\lambda = -3,\,-2,\,6$。

找到特征值和特征向量的意义远不止验证公式。把矩阵写成 `A = PΛP⁻¹` 后，`Aⁿ = PΛⁿP⁻¹`：原本需要 n 次 O(n³) 矩阵连乘，退化为对角线上各 λ 分别取 n 次方（O(n) 标量运算）。递归滤波器的稳定性就由此判断：系统矩阵所有 `|λᵢ| < 1` 时 `Aⁿ → 0`，滤波器收敛；任意一个 `|λᵢ| > 1` 则对应方向指数级发散。


## 符号入口：先看形状，再看运算

向量是 `(n,)`，矩阵是 `(n, n)`；`A @ v` 保持形状 `(n,)` 不变。本节要验证的关系是 `A @ v == λ * v`：左边是矩阵乘法，右边是标量缩放，两者应逐元素相等。

In [ ]:
import numpy as np
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
vals, vecs = np.linalg.eig(A)
print('特征值:', sorted(np.round(vals).astype(int)), ' (课件 -3,-2,6)')

## 动手观察：特征向量满足 `A @ v ≈ λ * v`

运行下面的代码，比对 `A @ v` 和 `λ * v` 每个分量是否相等。注意特征值为负时，特征向量方向被翻转；特征值绝对值大于 1 时，向量被拉伸。

In [ ]:
import numpy as np

# 验证特征分解：A = P D P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
D = np.diag(lam)
A_rec = P @ D @ np.linalg.inv(P)
print(f'特征值 λ = {np.round(lam, 4)}')
print(f'A =\n{A}')
print(f'P@D@P⁻¹ =\n{np.round(A_rec, 10)}')
print(f'重建误差 = {np.max(np.abs(A-A_rec)):.2e}')


## 代码实验：遍历特征向量，验证 `Av = λv`

对每个特征向量计算 `A @ v` 与 `λ * v` 的差，观察残差是否接近机器精度（约 1e-15）。

In [ ]:
import numpy as np

# 矩阵幂：用对角化 A^n = P D^n P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
Pinv = np.linalg.inv(P)
for n in [2, 5, 10]:
    A_n_diag = P @ np.diag(lam**n) @ Pinv      # 对角化法 O(N)
    A_n_iter = np.linalg.matrix_power(A, n)     # 直接法（参考）
    err = np.max(np.abs(A_n_diag - A_n_iter))
    print(f'A^{n:2d}[0,0] = {A_n_diag[0,0]:.2f}  误差 = {err:.2e}')


## 2. ✏️ 实现特征多项式（characteristic polynomial）取值 `char_poly(A, lam) = det(A − λI)`

在特征值处它应≈0。

**推理路线**：
1. 构造偏移矩阵 `A - lam * np.eye(len(A))`，把 A 的对角线每个元素减去 λ
2. 在真实特征值 λ 处，该矩阵把特征向量 v 映射到零向量，矩阵因此奇异
3. 奇异矩阵行列式（determinant）恰好为 0；非特征值处行列式明显非零（如 λ=5 时约 56）

**参考输入输出**：A 取课件 3×3 矩阵，`char_poly(A, -3)` ≈ 0，`char_poly(A, 5)` ≈ −384。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


写 `char_poly` 前明确三件事：
- 输入：矩阵 `A`（n×n）和候选特征值 `lam`（标量）
- 关键步骤：构造 `A - lam * np.eye(n)`，对其求行列式
- 返回：标量，在真实特征值处应接近 0

In [ ]:
def char_poly(A, lam):
    A = np.asarray(A, float)
    # ✏️ TODO: 返回 det(A - lam*I)
    return None


In [ ]:
for lam in (-3, -2, 6):
    print(f'char_poly(A, {lam}) =', round(char_poly(A, lam), 6))
assert all(abs(char_poly(A, lam)) < 1e-6 for lam in (-3,-2,6))
print('✅ 特征方程在三个特征值处都≈0，与课件一致。')

## 3. 对称矩阵对角化：`PᵀAP = D`（补充例题）

用**归一化**的特征向量按列拼成正交矩阵（orthogonal matrix） P，则 `PᵀAP` = 对角阵(对角线=特征值)。

课件特征向量：$v_1=(1,-1,-1),\ v_2=(0,1,-1),\ v_3=(2,1,1)$。


In [ ]:
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
v1=np.array([1,-1,-1.]); v2=np.array([0,1,-1.]); v3=np.array([2,1,1.])
P = np.column_stack([v/np.linalg.norm(v) for v in (v1,v2,v3)])
D = P.T @ A @ P
print('P 正交 (PᵀP=I):', np.allclose(P.T@P, np.eye(3)))
print('PᵀAP 对角线:', np.round(np.diag(D)).astype(int), '(课件 -3,-2,6)')
assert np.allclose(D - np.diag(np.diag(D)), 0, atol=1e-9)
print('✅ 成功对角化，与课件一致。')

**🔗 Aurora 连接**：`aurora/audio/` 中的递归滤波器稳定性检验就是对系统矩阵调用 `np.linalg.eig`，确认所有 `|λ| < 1`；PCA 对协方差矩阵做的是同一套特征分解，与 p6 SVD 互为近亲——`AᵀA = VΣ²Vᵀ` 即对 `AᵀA` 做特征分解，特征值恰好是奇异值的平方。

**补充例题对应**：特征方程、对称矩阵性质、正交对角化（orthogonal diagonalization） PᵀAP=D。


## 🎨 图示：对称矩阵的正交对角化 S = QΛQᵀ (补充例题)


In [ ]:
import numpy as np
from aurora.laviz import style, show_factorization
style()
S=np.array([[3.,3,3],[3,-1,1],[3,1,-1]]); w,Q=np.linalg.eigh(S)
show_factorization(S,[Q,np.diag(w),Q.T],['Q','Λ','Qᵀ'],
                   modes=['#2A9D8F','#E9C46A','#2A9D8F'], title='S = Q Λ Qᵀ');

In [ ]:
import numpy as np

# 参数实验：改变矩阵参数观察特征值变化
print(f'{'参数 c':>8}  {'λ₁':>10}  {'λ₂':>10}  {'|λ₁/λ₂|':>10}')
for c in [0.5, 1.0, 2.0, 4.0]:
    A = np.array([[3., c],[1., 2.]])
    lam = np.linalg.eigvals(A)
    lam = sorted(lam, key=abs, reverse=True)
    ratio = abs(lam[0]/lam[1]) if abs(lam[1]) > 1e-10 else float('inf')
    print(f'{c:>8.1f}  {lam[0]:>+10.4f}  {lam[1]:>+10.4f}  {ratio:>10.4f}')
print('→ 两特征值相差越大，幂迭代收敛越快（主导方向越突出）。')


## 参数实验：稳定边界的两侧

构造特征值分别为 0.9 和 1.1 的矩阵，计算 A^50，观察两个方向的长期行为：
- `0.9^50 ≈ 0.005`：该特征方向已衰减到接近零（稳定）
- `1.1^50 ≈ 117`：该特征方向指数爆炸（不稳定）

仅凭特征值模是否越过 1.0 这条线，就能预判系统长期行为。Aurora 状态空间模型的稳定性保证就是把系统矩阵所有特征值的模约束在单位圆内。

## 本课收束

现在可以用 `np.linalg.eig(A)` 取得特征值数组和特征向量矩阵 `P`，并用 `char_poly(A, lam)` 验证任意 `λ` 是否为真正的特征值。这对应 Aurora 状态空间模型的稳定性检验：把系统矩阵传入特征分解，若所有 `|λ| < 1`，递归滤波器就收敛。下一节（**L18**）补全可逆性判据，使 `A = PΛP⁻¹` 可以精确还原。